In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
import torch
import torch.nn as nn

seed = 1
torch.manual_seed(seed)


In [8]:
!pip install unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 3.8 MB/s eta 0:00:00a 0:00:01


In [9]:
import os
import re
import nltk
from unidecode import unidecode  # sửa lại tên thư viện

# tải stopwords
nltk.download('stopwords')


[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [10]:
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [20]:
import pandas as pd

# Đường dẫn file Excel
data_path = "/kaggle/input/dataset-lstm/Data4.xlsx"

# Đọc file Excel
df = pd.read_excel(data_path)

# Lưu thành file CSV (không ghi chỉ số index)
csv_path = "/kaggle/working/Data4.csv"
df.to_csv(csv_path, index=False)

print("Đã chuyển sang CSV:", csv_path)


Đã chuyển sang CSV: /kaggle/working/Data4.csv


In [23]:
df = pd.read_csv("/kaggle/working/Data4.csv")

In [24]:
print(df.columns)


Index(['question', 'answer'], dtype='object')


In [27]:
!pip install pyvi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 48.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 39.0 MB/s eta 0:00:00


In [28]:
from pyvi import ViTokenizer

In [29]:
classes = {
    class_name: idx for idx, class_name in enumerate(df['answer'].unique())
}


In [34]:
# --- Đọc stopwords từ file txt ---
stopwords_path = "/kaggle/input/stopwords-vietnamese/vietnamese-stopwords.txt"
with open(stopwords_path, "r", encoding="utf-8") as f:
    vietnamese_stop_words = f.read().splitlines()

# --- Hàm loại bỏ stopwords ---
def remove_stopwords(text, stopwords):
    if not isinstance(text, str):
        return text
    words = text.split()
    filtered_words = [w for w in words if w not in stopwords]
    return " ".join(filtered_words)

# --- Áp dụng vào một cột văn bản (ví dụ cột 'Text') ---
df["question"] = df["question"].apply(lambda x: remove_stopwords(x, vietnamese_stop_words))

In [35]:
sentence = "Tôi và anh ấy đang đi trong công viên để tập thể dục"

cleaned = remove_stopwords(sentence, vietnamese_stop_words)
print(cleaned)


Tôi đi công viên tập thể dục


In [48]:
import re
from pyvi.ViTokenizer import tokenize

# --- 1. Chuẩn hóa chữ thường ---
def to_lowercase(text):
    return text.lower()

# --- 2. Loại bỏ số ---
def remove_numbers(text):
    return re.sub(r'\d+', '', text)

# --- 3. Loại bỏ dấu câu ---
def remove_punctuation(text):
    return re.sub(r'[^\w\s]', '', text)

# --- 4. Chuẩn hóa khoảng trắng ---
def normalize_whitespace(text):
    return re.sub(r'\s+', ' ', text).strip()

# --- 5. Tách từ với PyVi ---
def word_tokenize(text):
    return tokenize(text)

# --- Hàm tổng hợp normalize ---
def text_normalize(text, stopwords):
    text = to_lowercase(text)
    text = remove_numbers(text)
    text = remove_punctuation(text)
    text = normalize_whitespace(text)
    text = word_tokenize(text)
    text = remove_stopwords(text, stopwords)
    return text


In [49]:
df['question'] = df['question'].apply(lambda x: text_normalize(str(x), vietnamese_stop_words))

# --- Tạo vocab từ cột 'answer' ---
vocab = []
for sentence in df['answer'].tolist():
    tokens = sentence.split()
    for token in tokens:
        if token not in vocab:
            vocab.append(token)

vocab.append('UNK')
vocab.append('PAD')
word_to_idx = {word: idx for idx, word in enumerate(vocab)}
vocab_size = len(vocab)







In [50]:
# --- Hàm chuyển text thành sequence ---
def transform(text, word_to_idx, max_seq_len):
    tokens = []
    for w in text.split():
        w_id = word_to_idx[w] if w in word_to_idx else word_to_idx['UNK']
        tokens.append(w_id)
    if len(tokens) < max_seq_len:
        tokens += [word_to_idx['PAD']] * (max_seq_len - len(tokens))
    elif len(tokens) > max_seq_len:
        tokens = tokens[:max_seq_len]
    return tokens

In [51]:
# --- Chia dữ liệu train / val / test ---
val_size = 0.2   # 20% validation
test_size = 0.125  # 12.5% test
seed = 42
is_shuffle = True

answer = df['answer'].tolist()
question = df['question'].tolist()

# Train + Temp
X_train, X_temp, y_train, y_temp = train_test_split(
    answer, question,
    test_size=(val_size + test_size),
    random_state=seed,
    shuffle=is_shuffle
)


In [52]:

# Validation + Test
relative_test_size = test_size / (val_size + test_size)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=relative_test_size,
    random_state=seed,
    shuffle=is_shuffle
)


In [53]:
# --- Dataset class ---
class Generate_Answer(Dataset):
    def __init__(self, X, y, word_to_idx, max_seq_len, transform=None):
        self.answer = X
        self.question = y
        self.word_to_idx = word_to_idx
        self.max_seq_len = max_seq_len
        self.transform = transform

    def __len__(self):
        return len(self.answer)

    def __getitem__(self, idx):
        question = self.question[idx]
        answer = self.answer[idx]
        if self.transform:
            answer = self.transform(
                answer,
                self.word_to_idx,
                self.max_seq_len
            )
            answer = torch.tensor(answer)
        return answer, question


In [54]:
# --- Tạo dataset ---
max_seq_len = 32
train_dataset = Generate_Answer(X_train, y_train, word_to_idx, max_seq_len, transform=transform)
val_dataset = Generate_Answer(X_val, y_val, word_to_idx, max_seq_len, transform=transform)
test_dataset = Generate_Answer(X_test, y_test, word_to_idx, max_seq_len, transform=transform)

# --- Tạo DataLoader ---
train_batch_size = 128
test_batch_size = 8

train_loader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=test_batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=False)

In [63]:
import torch
import torch.nn as nn

class Answer_Classifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, n_layers, n_classes, dropout_prob):
        super(Answer_Classifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=word_to_idx['PAD'])
        self.rnn = nn.RNN(embedding_dim, hidden_size, n_layers, batch_first=True)
        self.norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout_prob)
        self.fc1 = nn.Linear(hidden_size, 16)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(16, n_classes)
    
    def forward(self, x):
        x = self.embedding(x)         # [batch, seq_len, emb_dim]
        x, hn = self.rnn(x)           # [batch, seq_len, hidden_size]
        x = x[:, -1, :]               # lấy output cuối cùng
        x = self.norm(x)
        x = self.dropout(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)               # sửa lại
        return x

# --- Khởi tạo model ---
n_classes = len(list(classes.keys()))
embedding_dim = 64
hidden_size = 64
n_layers = 2
dropout_prob = 0.2

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = Answer_Classifier(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_size=hidden_size,
    n_layers=n_layers,
    n_classes=n_classes,
    dropout_prob=dropout_prob
).to(device)


In [67]:
lr = 1e-4
epochs = 50
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr = lr
)

In [68]:
def evaluate(model, data_loader, criterion, device):
    model.eval()
    losses = []
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, question in data_loader:
            inputs = inputs.to(device)
            labels = torch.tensor([classes[q] for q in question], dtype=torch.long).to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            losses.append(loss.item())

            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = sum(losses) / len(losses)
    acc = correct / total if total > 0 else 0
    return avg_loss, acc


def fit(model, train_loader, val_loader, criterion, optimizer, device, epochs):
    train_losses = []
    val_losses = []
    val_accs = []

    for epoch in range(epochs):
        batch_train_losses = []
        model.train()

        for idx, (inputs, question) in enumerate(train_loader):
            inputs = inputs.to(device)
            labels = torch.tensor([classes[q] for q in question], dtype=torch.long).to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            batch_train_losses.append(loss.item())

        # --- Tính loss trung bình train ---
        train_loss = sum(batch_train_losses) / len(batch_train_losses)
        train_losses.append(train_loss)

        # --- Evaluate ---
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        print(f'Epoch {epoch+1}/{epochs}: Train loss = {train_loss:.4f}, Val loss = {val_loss:.4f}, Val acc = {val_acc:.4f}')

    return train_losses, val_losses, val_accs


In [69]:
train_losses, val_losses = fit(
    model, train_loader, val_loader, criterion, optimizer, device, epochs
)

KeyError: 'ngành httt trường đối_tượng tuyển_sinh'

In [94]:
!pip install underthesea

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 45.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.6/978.6 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 83.7 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.2.2
    Uninstalling scikit-learn-1.2.2:
      Successfully uninstalled scikit-learn-1.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
category-encoders 2.7.0 requires scikit-learn<1.6.0,>=1.0.0, but you have scikit-learn 1.7.2 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
sklearn-compat 0.1.3 requires scikit-learn<1.7,>=1.2, but you have scikit-learn 1.7.2 which is incompatible.


In [95]:
import re
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from underthesea import word_tokenize  # tokenizer cho tiếng Việt

# =========================
# 1. Load & Preprocess Data
# =========================
df = pd.read_excel("/kaggle/input/dataset-lstm/Data4.xlsx")   # thay bằng đường dẫn file excel
df = df.dropna()

# encode nhãn (answer)
classes = {label: idx for idx, label in enumerate(df['answer'].unique())}
inv_classes = {v: k for k, v in classes.items()}
df['label'] = df['answer'].map(classes)

# text preprocessing
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)  # bỏ dấu câu
    text = " ".join(word_tokenize(text))  # tách từ tiếng Việt
    return text

df['question'] = df['question'].apply(clean_text)

# build vocab
all_tokens = set()
for q in df['question']:
    for tok in q.split():
        all_tokens.add(tok)

vocab = {word: idx+2 for idx, word in enumerate(all_tokens)}
vocab["<pad>"] = 0
vocab["<unk>"] = 1
vocab_size = len(vocab)

# =========================
# 2. Dataset & DataLoader
# =========================
class QADataset(Dataset):
    def __init__(self, questions, labels, vocab, max_len=30):
        self.questions = questions
        self.labels = labels
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        tokens = self.questions[idx].split()
        ids = [self.vocab.get(tok, self.vocab["<unk>"]) for tok in tokens]
        if len(ids) < self.max_len:
            ids += [self.vocab["<pad>"]] * (self.max_len - len(ids))
        else:
            ids = ids[:self.max_len]
        return torch.tensor(ids, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

X_train, X_val, y_train, y_val = train_test_split(df['question'].tolist(), df['label'].tolist(),
                                                  test_size=0.2, random_state=42)

train_dataset = QADataset(X_train, y_train, vocab)
val_dataset = QADataset(X_val, y_val, vocab)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

# =========================
# 3. Model
# =========================
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, n_layers, n_classes, dropout_prob):
        super(LSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, num_layers=n_layers,
                            batch_first=True, dropout=dropout_prob, bidirectional=True)
        self.fc = nn.Linear(hidden_size * 2, n_classes)
        self.dropout = nn.Dropout(dropout_prob)

    def forward(self, x):
        embedded = self.embedding(x)          # (B, L, E)
        lstm_out, _ = self.lstm(embedded)     # (B, L, 2H)
        mean_pool = torch.mean(lstm_out, dim=1)  # trung bình theo chiều sequence
        out = self.dropout(mean_pool)
        return self.fc(out)

# =========================
# 4. Train & Evaluate
# =========================
def evaluate(model, data_loader, criterion, device):
    model.eval()
    losses, correct, total = [], 0, 0
    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            losses.append(loss.item())
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return sum(losses) / len(losses), correct / total if total > 0 else 0

def fit(model, train_loader, val_loader, criterion, optimizer, device, epochs=50, patience=5):
    best_val_loss = float("inf")
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        train_losses = []
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        train_loss = sum(train_losses) / len(train_losses)
        print(f"Epoch {epoch+1}/{epochs}: Train={train_loss:.4f}, Val={val_loss:.4f}, Acc={val_acc:.4f}")

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), "best_model.pt")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered!")
                break

# =========================
# 5. Run training
# =========================
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = LSTMClassifier(
    vocab_size=vocab_size,
    embedding_dim=128,
    hidden_size=128,
    n_layers=2,
    n_classes=len(classes),
    dropout_prob=0.4
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

fit(model, train_loader, val_loader, criterion, optimizer, device, epochs=50)

# Load best model
model.load_state_dict(torch.load("best_model.pt"))

# =========================
# 6. Dự đoán thử
# =========================
def predict(model, text, max_len=30):
    model.eval()
    text = clean_text(text)
    tokens = text.split()
    ids = [vocab.get(tok, vocab["<unk>"]) for tok in tokens]
    if len(ids) < max_len:
        ids += [vocab["<pad>"]] * (max_len - len(ids))
    else:
        ids = ids[:max_len]
    x = torch.tensor([ids], dtype=torch.long).to(device)
    with torch.no_grad():
        outputs = model(x)
        pred = torch.argmax(outputs, dim=1).item()
    return inv_classes[pred]

print(predict(model, "Xin chào bot"))
print(predict(model, "Trọ ở đâu quanh trường"))


Epoch 1/50: Train=4.2635, Val=3.6359, Acc=0.1189
Epoch 2/50: Train=3.2450, Val=2.6059, Acc=0.3471
Epoch 3/50: Train=2.2590, Val=1.7666, Acc=0.5194
Epoch 4/50: Train=1.4793, Val=1.1761, Acc=0.7184
Epoch 5/50: Train=0.9177, Val=0.8095, Acc=0.8277
Epoch 6/50: Train=0.5794, Val=0.4886, Acc=0.9199
Epoch 7/50: Train=0.3683, Val=0.3851, Acc=0.9417
Epoch 8/50: Train=0.2835, Val=0.3242, Acc=0.9442
Epoch 9/50: Train=0.1923, Val=0.2656, Acc=0.9587
Epoch 10/50: Train=0.1542, Val=0.2641, Acc=0.9563
Epoch 11/50: Train=0.1641, Val=0.2442, Acc=0.9539
Epoch 12/50: Train=0.0972, Val=0.2204, Acc=0.9660
Epoch 13/50: Train=0.0813, Val=0.2264, Acc=0.9660
Epoch 14/50: Train=0.0773, Val=0.2566, Acc=0.9612
Epoch 15/50: Train=0.0763, Val=0.2061, Acc=0.9660
Epoch 16/50: Train=0.0465, Val=0.2200, Acc=0.9636
Epoch 17/50: Train=0.0536, Val=0.2333, Acc=0.9636
Epoch 18/50: Train=0.0522, Val=0.2051, Acc=0.9684
Epoch 19/50: Train=0.0377, Val=0.1965, Acc=0.9684
Epoch 20/50: Train=0.0298, Val=0.2028, Acc=0.9684
Epoch 21/

In [ ]:
print(predict(model, "code ra bot, code ra bot, viết ra bot"))
